# 02. Từ khóa đặc trưng theo nhãn (ViHSD, chỉ tập train)

> **Cảnh báo nội dung:** output chứa từ thô tục. **Clear All Outputs trước khi commit.**

Phương pháp: log-odds có prior Dirichlet (Monroe et al., 2008), code ở `src/keywords.py`.
Mỗi token được đếm tối đa 1 lần trong một bình luận. `z > 1.96` ~ khác biệt có ý nghĩa ở mức khoảng 95%.

In [ ]:
import sys

if ".." not in sys.path:
    sys.path.insert(0, "..")

import pandas as pd

from src.data import LABEL_COL, LABEL_NAMES, load_vihsd
from src.keywords import count_tokens_by_label, log_odds_by_label

PRIOR_STRENGTH = 1000.0
MIN_COUNT = 5
TOP_K = 30

train = load_vihsd("train", "../data/raw/vihsd")
docs_per_label = train[LABEL_COL].value_counts().sort_index()
docs_per_label

In [ ]:
def keyword_table(n: int) -> pd.DataFrame:
    counts = count_tokens_by_label(train, n=n)
    result = log_odds_by_label(counts, prior_strength=PRIOR_STRENGTH, min_count=MIN_COUNT)
    result["share_in_label"] = result["count"] / result["label"].map(docs_per_label)
    return result


def show_top(result: pd.DataFrame, by: str) -> None:
    for label, name in LABEL_NAMES.items():
        top = result[result["label"] == label].nlargest(TOP_K, by)
        print(f"=== {name} (top {TOP_K} theo {by}) ===")
        print(
            top[["token", "count", "count_rest", "share_in_label", "z"]]
            .round(3)
            .to_string(index=False)
        )
        print()


unigrams = keyword_table(n=1)
bigrams = keyword_table(n=2)
print("Số âm tiết khác nhau (sau min_count):", unigrams["token"].nunique())
print("Số bigram khác nhau (sau min_count):", bigrams["token"].nunique())

## So sánh: tần suất thô vs log-odds (âm tiết)

Tần suất thô: top của mọi nhãn gần như giống nhau (từ phổ biến). Log-odds: top khác nhau giữa các nhãn.

In [ ]:
show_top(unigrams, by="count")

In [ ]:
show_top(unigrams, by="z")

## Bigram (log-odds)

In [ ]:
show_top(bigrams, by="z")

## Tổng kết dạng số (dùng cho data/README.md, không chứa nguyên văn từ)

In [ ]:
summary = []
for kind, result in [("unigram", unigrams), ("bigram", bigrams)]:
    for label, name in LABEL_NAMES.items():
        z = result.loc[result["label"] == label, "z"]
        summary.append(
            {
                "kind": kind,
                "label": name,
                "tokens": len(z),
                "z>1.96": int((z > 1.96).sum()),
                "z>5": int((z > 5).sum()),
                "max_z": round(float(z.max()), 2),
            }
        )
pd.DataFrame(summary)